# LLM Law Learning Machine: pipeline completo en Colab

Corre **el repo** (el grafo de `src/`) de punta a punta: trae el código, monta el corpus y los índices desde Drive,
levanta el decoder con Ollama, genera las respuestas con `main.py` y las mide con el `evaluate.py` oficial.

**Lo que tiene que estar en Drive** (`MyDrive/hackathon_vectorial`, la misma carpeta de siempre):
- `corpus.db` (la de los offsets, la que dejó `exportar_entrega.py --guardar-offsets`) o `corpus_v1.zip`,
- `indices/index_sin_sentencias/` e `indices/index_juris/` (con `dense.faiss`, `bm25/`, `chunk_ids.json`, `info.json`).

**Secretos de Colab** (ícono de la llave, con *Acceso al notebook* activado):
- `GITHUB_TOKEN`: un token de GitHub con permiso de lectura al repo (el repo es privado).
- `OPENROUTER_API_KEY`: la llave del juez, solo para `evaluate.py --ragas`. No se guarda en ningún archivo.

**Entorno:** *Entorno de ejecución → Cambiar tipo → GPU L4* (o T4), luego *Ejecutar todo*.
La primera vez baja el modelo (~9 GB) y lo deja en Drive; las siguientes arranca en segundos.

In [ ]:
# ===== Configuracion =====
import os
REPO = "NBaqueroO/LLM-Law-Learning-Machine"
RAMA = "integracion"                # la rama que se quiere correr (sale de graph_development)
MODELO = "hf.co/cstr/salamandra-7b-instruct-GGUF"            
CONTEXTO = 8192                     # tokens; Ollama trae 4096 por defecto y 10 pasajes no caben
PARALELO = 4                        # preguntas a la vez en la GPU (sube si sobra memoria)
SPLIT = "sample"                    # "sample" para medir; "test" el sabado
ARGS_EXTRA = ["--concurrencia", str(PARALELO), "--sin-evaluar"]   # el sabado, en cada GPU: + ["--parte", "1/2"] o ["--parte", "2/2"]
USAR_JUEZ = True                    # evaluate.py --ragas (gasta la llave)
CARPETA = "/content/drive/MyDrive/hackathon_vectorial"
EXPERIMENTO = "base"                # nombre corto: va en el nombre del resultado en Drive (entregas_grafo/)

# Perillas del sistema (variables de entorno que lee src/config.py). Vacio = el valor del repo.
AJUSTES = {
    "UMBRAL_SCORE": "",             # 0.30: debajo, reformula una vez
    "PISO_ABSTENCION": "",          # 0.05: debajo (tras reformular), el texto libre se abstiene
    "CUPO_JURIS": "",               # 0.3: parte del top-10 para sentencias
    "K_CANDIDATOS": "",             # 50: candidatos por buscador antes de fusionar
    "RERANKER": "",                 # "BAAI/bge-reranker-v2-m3" para prenderlo (mas lento)
    "AREA_EN_CONSULTA": "",         # "0" apaga sumar los codigos del area a la 1.a busqueda
    "CITAR_RECUPERADAS": "",        # "0" apaga citar las normas del top-10 que el modelo no nombro
    "PROMPT_EVALUACION": "",        # "1": el prompt le explica al modelo como se califica
    "LLM_MAX_TOKENS": "",           # 1200
    "LLM_TIMEOUT": "",              # 600 s
}
for k, v in AJUSTES.items():
    if v != "":
        os.environ[k] = str(v)
    else:
        os.environ.pop(k, None)

In [ ]:
# ===== Drive, repo y dependencias =====
import os, sys, json, glob, shutil, sqlite3, subprocess, time, zipfile
from pathlib import Path
os.environ["JAX_PLATFORMS"] = "cpu"   # bm25s importa JAX en Colab y JAX se queda con el 75 % de la GPU

from google.colab import drive, userdata
drive.mount("/content/drive")
WORK = Path(CARPETA)
RAPIDO = Path("/content/rapido"); RAPIDO.mkdir(exist_ok=True)

import base64
token = userdata.get("GITHUB_TOKEN").strip()
RAIZ = Path("/content") / REPO.split("/")[1]
url = f"https://github.com/{REPO}.git"
# el token va en una variable de entorno de git, nunca en la URL ni en el comando: asi no sale en errores
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
genv = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}

def git(*args):
    r = subprocess.run(["git", *args], env=genv, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {args[0]} fallo:\n{r.stderr.replace(token, '***')}\n"
                           "Si dice 'not found' o 'Authentication failed': el token debe ser *classic* con permiso "
                           "'repo' (los github_pat_ no ven repos privados de otra persona).")
    return r.stdout

if (RAIZ / ".git").exists():
    git("-C", str(RAIZ), "fetch", "-q", "origin", RAMA)
    git("-C", str(RAIZ), "checkout", "-q", "-B", RAMA, f"origin/{RAMA}")
else:
    git("clone", "-q", "-b", RAMA, url, str(RAIZ))
print(subprocess.run(["git", "-C", str(RAIZ), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements.txt"),
                "-r", str(RAIZ / "scripts" / "requirements-evaluador.txt")], check=True)
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("repo listo en", RAIZ)

In [ ]:
# ===== corpus.db e indices =====
def db_sana(p):
    try:
        con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
        ok = con.execute("PRAGMA quick_check").fetchone()[0] == "ok"
        con.close()
        return ok
    except sqlite3.Error:
        return False

DB = RAPIDO / "corpus.db"
# Se copia una vez por sesion al disco local (SQLite sobre Drive es lento). La revision se hace
# sobre la copia local: revisarla en Drive obligaba a leer los GB dos veces.
if not (DB.exists() and db_sana(DB)):
    tmp = RAPIDO / "corpus.db.tmp"
    t = time.time()
    if (WORK / "corpus.db").exists():
        print(f"copiando corpus.db de Drive ({(WORK / 'corpus.db').stat().st_size / 1e9:.1f} GB)...")
        shutil.copyfile(WORK / "corpus.db", tmp)
    else:
        z = WORK / "corpus_v1.zip"
        assert z.exists(), f"No encuentro {WORK / 'corpus.db'} ni {z}"
        print(f"descomprimiendo {z}...")
        with zipfile.ZipFile(z) as zz, zz.open(next(n for n in zz.namelist() if n.endswith("corpus.db"))) as src, \
                open(tmp, "wb") as dst:
            shutil.copyfileobj(src, dst, 1 << 24)
    os.replace(tmp, DB)
    print(f"listo en {(time.time() - t) / 60:.1f} min")
assert db_sana(DB), f"{DB} esta danada: borra corpus.db de Drive y vuelve a subirla"

def indice(nombre):
    hall = sorted(glob.glob(str(WORK / "**" / nombre / "dense.faiss"), recursive=True))
    assert hall, f"No encuentro {nombre}/dense.faiss dentro de {WORK}"
    return Path(hall[0]).parent

os.environ["CORPUS_DB"] = str(DB)
os.environ["INDEX_NORMAS"] = str(indice("index_sin_sentencias"))
os.environ["INDEX_JURIS"] = str(indice("index_juris"))
# index_manifest.json (sha256 de corpus.db y de los indices): se arma una vez y queda en Drive
os.environ["INDEX_MANIFEST"] = str(Path(os.environ["INDEX_NORMAS"]).parent / "index_manifest.json")
if not Path(os.environ["INDEX_MANIFEST"]).exists():
    print("armando index_manifest.json (una sola vez, lee todos los archivos)...")
    subprocess.run([sys.executable, "-m", "src.indexing.build_index", "--solo-manifiesto", "--db", str(DB)], check=True)
# Solo importan los fragmentos que estan en los indices (los demas nunca salen en pasajes_recuperados)
ids = [c for n in ("INDEX_NORMAS", "INDEX_JURIS")
       for c in json.load(open(Path(os.environ[n]) / "chunk_ids.json", encoding="utf-8"))]
con = sqlite3.connect(DB)
con.execute("CREATE TEMP TABLE ix(chunk_id TEXT PRIMARY KEY)")
con.executemany("INSERT OR IGNORE INTO ix VALUES (?)", ((str(c),) for c in ids))
total = con.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
sin_offsets = con.execute("SELECT COUNT(*) FROM chunks JOIN ix USING(chunk_id) WHERE inicio IS NULL").fetchone()[0]
con.close()
print(f"fragmentos: {total} en corpus.db, {len(set(ids))} en los indices, {sin_offsets} de esos sin inicio/fin")
print("corpus.db:", DB, "\nindices:", os.environ["INDEX_NORMAS"], "+", os.environ["INDEX_JURIS"])
if sin_offsets:
    print(f"OJO: {sin_offsets} fragmentos indexados sin inicio/fin. Para medir da igual; para la entrega sube a Drive "
          "la corpus.db de exportar_entrega.py --guardar-offsets.")

In [ ]:
# ===== Decoder: Ollama con el modelo guardado en Drive =====
if shutil.which("ollama") is None:
    # el instalador de Ollama ahora baja un .tar.zst: sin zstd falla con "exit status 1"
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", shell=True,
                   stdout=subprocess.DEVNULL)
    r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
    if r.returncode or shutil.which("ollama") is None:
        raise RuntimeError("no se pudo instalar Ollama:\n" + (r.stderr or r.stdout)[-2000:])
os.environ["OLLAMA_MODELS"] = str(WORK / "ollama")         # el modelo se baja una vez y queda en Drive
os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXTO)
os.environ["OLLAMA_NUM_PARALLEL"] = str(PARALELO)
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"                     # que no descargue el modelo entre preguntas
import urllib.request

def ollama_vivo():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not ollama_vivo():
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            break
        time.sleep(1)
assert ollama_vivo(), "Ollama no arranco: mira /content/ollama.log"
subprocess.run(["ollama", "pull", MODELO], check=True)

# lo que lee src/config.py
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_MODELO"] = MODELO

# La primera vez Ollama carga ~9 GB desde Drive a la GPU y puede tardar varios minutos:
# se precalienta con una espera larga para que la prueba de abajo no se corte por timeout.
t = time.time()
print("cargando el modelo en la GPU (la primera vez puede tardar unos minutos)...")
req = urllib.request.Request("http://localhost:11434/api/generate", method="POST",
                             data=json.dumps({"model": MODELO, "prompt": "hola", "stream": False,
                                              "keep_alive": -1}).encode(),
                             headers={"Content-Type": "application/json"})
urllib.request.urlopen(req, timeout=1800).read()
print(f"modelo cargado en {time.time() - t:.0f} s")

from src.generation.llm_engine import texto_libre
t = time.time()
print(texto_libre("Responde en una frase.", "¿Qué norma es el Código General del Proceso?"), f"({time.time() - t:.1f} s)")

In [ ]:
# ===== Pruebas rapidas del repo (con el corpus real) =====
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])

In [ ]:
# ===== La arquitectura: el grafo compilado y el recorrido de una pregunta =====
# El diagrama sale del grafo que de verdad corre (src/graph/workflow.py), no de un dibujo aparte.
# tests/test_arquitectura.py falla si alguna arista deja de coincidir con el diagrama del equipo.
from IPython.display import Image, display
from src.graph.workflow import construir_grafo
try:
    display(Image(construir_grafo().get_graph().draw_mermaid_png()))
except Exception as e:   # sin internet para mermaid.ink: el texto se pega en mermaid.live
    print(construir_grafo().get_graph().draw_mermaid(), "\n(", e, ")")

# Una pregunta real, nodo por nodo, con lo que cada nodo deja en el estado
primera = json.loads(open("data/sample_50.jsonl", encoding="utf-8").readline())["id"]
correr = lambda args: subprocess.run([sys.executable, "-u", *map(str, args)])
correr(["main.py", "--split", "sample", "--ids", primera, "--ruta"])

In [ ]:
# ===== Correr el grafo: main.py (si se corta, volver a correr la celda sigue donde iba) =====
def correr(args):
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)], text=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, encoding="utf-8", errors="replace")
    try:
        for linea in p.stdout:
            print(linea, end="")
        return p.wait()
    finally:
        if p.poll() is None:
            p.kill()

t = time.time()
cod = correr(["main.py", "--split", SPLIT, *ARGS_EXTRA])
print(f"\nmain.py termino con codigo {cod} en {(time.time() - t) / 60:.1f} min")

SALIDA = Path("outputs") / f"{'sample_50' if SPLIT == 'sample' else 'submissions'}.jsonl"
hallados = sorted((p for p in Path("outputs").glob("*.jsonl") if not p.name.endswith(".trazas.jsonl")),
                  key=lambda p: p.stat().st_mtime) if Path("outputs").exists() else []
if not SALIDA.exists() and hallados:
    SALIDA = hallados[-1]
assert SALIDA.exists(), "main.py no dejo ningun .jsonl en outputs/ (¿ya esta implementado?)"
n = sum(1 for _ in open(SALIDA, encoding="utf-8"))
print(f"{SALIDA}: {n} respuestas")
copia = WORK / "entregas_grafo" / f"{time.strftime('%Y%m%d_%H%M')}_{EXPERIMENTO}_{SALIDA.name}"
copia.parent.mkdir(exist_ok=True)
shutil.copy(SALIDA, copia)
trz = SALIDA.with_name(SALIDA.stem + ".trazas.jsonl")
if trz.exists():
    shutil.copy(trz, copia.with_name(copia.stem + ".trazas.jsonl"))
json.dump({k: v for k, v in AJUSTES.items() if v != ""} | {"MODELO": MODELO, "RAMA": RAMA},
          open(copia.with_name(copia.stem + ".ajustes.json"), "w"), indent=1)
print("copia en Drive:", copia)

In [ ]:
# ===== Evaluador oficial =====
env = dict(os.environ)
args = [sys.executable, "scripts/evaluate.py", "--submission", str(SALIDA), "--split", SPLIT,
        "--out", str(WORK / "entregas_grafo" / (copia.stem + ".eval.json"))]
if USAR_JUEZ and SPLIT == "sample":
    env["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    args.append("--ragas")
r = subprocess.run(args, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-6000:] or r.stderr[-6000:])
print("Referencias: linea base agentes.ipynb 47,05/80; primer grafo (2026-10-01) 49,75/80")

## Opcional: comparar modelos (para dejar corriendo)

Corre `sample_50` con cada modelo de la lista, uno tras otro, y deja en Drive
(`entregas_grafo/modelos/`) las respuestas, las trazas, el puntaje y un `resumen.md` con la tabla.
Cada modelo tarda ~30 min en L4 (bajarlo, 50 preguntas y el juez). **Deja la pestaña abierta y el
computador sin suspender**: si Colab se desconecta, vuelve a correr las celdas 1 a 4 y esta, y salta
los modelos que ya terminaron.
Los modelos de prueba se guardan en el disco de Colab, no en Drive (no cabrían).

In [ ]:
# ===== Comparar modelos =====
# nombre corto -> etiqueta de Ollama (o lista de etiquetas para probar en orden si alguna no existe)
MODELOS = {
    "qwen3_8b_q8": "qwen3:8b-q8_0",                        # el actual (con el repo de hoy)
    "llama31_8b_q8": "llama3.1:8b-instruct-q8_0",           # Meta, multilingue
    "qwen25_7b_q8": "qwen2.5:7b-instruct-q8_0",             # la generacion anterior de Qwen
    "aya_8b_q8": "aya-expanse:8b-q8_0",                     # Cohere, fuerte en espanol (licencia no comercial)
    "salamandra_7b_q8": [                                   # BSC (familia ALIA), entrenado con mucho espanol
        # con "hf.co/" Ollama falla ("realm host huggingface.co does not match"): va el dominio completo
        "huggingface.co/BSC-LT/salamandra-7b-instruct-GGUF:Q8_0",
        "huggingface.co/mradermacher/salamandra-7b-instruct-GGUF:Q8_0",
    ],
}
DESTINO = WORK / "entregas_grafo" / "modelos"
DESTINO.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/ollama_modelos")

# Ollama de nuevo, con los modelos en el disco local (no necesita la celda del decoder)
import urllib.request
if shutil.which("ollama") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", shell=True,
                   stdout=subprocess.DEVNULL)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True)
os.environ.update({"OLLAMA_CONTEXT_LENGTH": str(CONTEXTO), "OLLAMA_NUM_PARALLEL": str(PARALELO),
                   "OLLAMA_KEEP_ALIVE": "-1", "LLM_TIMEOUT": os.environ.get("LLM_TIMEOUT") or "600"})

def ollama_vivo():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

subprocess.run(["pkill", "-f", "ollama serve"])
time.sleep(3)
env_ollama = {**os.environ, "OLLAMA_MODELS": str(LOCAL)}
subprocess.Popen(["ollama", "serve"], env=env_ollama, stdout=open("/content/ollama.log", "a"),
                 stderr=subprocess.STDOUT)
for _ in range(60):
    if ollama_vivo():
        break
    time.sleep(1)
assert ollama_vivo(), "Ollama no arranco: mira /content/ollama.log"

def bajar(etiquetas):
    for e in ([etiquetas] if isinstance(etiquetas, str) else etiquetas):
        r = subprocess.run(["ollama", "pull", e], env=env_ollama, capture_output=True, text=True)
        if r.returncode == 0:
            return e
        print(f"  no se pudo bajar {e}: {(r.stderr or r.stdout).strip()[-300:]}")
    return None

for nombre, etiquetas in MODELOS.items():
    ev = DESTINO / f"{nombre}.eval.json"
    if ev.exists():
        print(f"== {nombre}: ya estaba, lo salto")
        continue
    print(f"\n== {nombre}")
    etiqueta = bajar(etiquetas)
    if etiqueta is None:
        print(f"  {nombre}: no se pudo bajar ninguna version, sigo con el siguiente")
        continue
    salida = Path("outputs") / f"modelo_{nombre}.jsonl"
    env = {**os.environ, "LLM_MODELO": etiqueta, "LLM_BASE_URL": "http://localhost:11434/v1"}
    t = time.time()
    p = subprocess.run([sys.executable, "-u", "main.py", "--split", "sample", "--salida", str(salida),
                        "--concurrencia", str(PARALELO), "--sin-evaluar"], env=env)
    minutos = (time.time() - t) / 60
    if not salida.exists():
        print(f"  {nombre}: main.py no dejo respuestas (codigo {p.returncode}), sigo")
        continue
    args = [sys.executable, "scripts/evaluate.py", "--submission", str(salida), "--split", "sample", "--out", str(ev)]
    env_eval = dict(os.environ)
    if USAR_JUEZ:
        env_eval["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
        args.append("--ragas")
    r = subprocess.run(args, env=env_eval, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if not ev.exists():
        print(f"  {nombre}: el evaluador fallo:\n{(r.stderr or r.stdout)[-1500:]}")
    for f in (salida, salida.with_name(salida.stem + ".trazas.jsonl")):
        if f.exists():
            shutil.copy(f, DESTINO / f.name)
    json.dump({"modelo": etiqueta, "minutos": round(minutos, 1), "ajustes": {k: v for k, v in AJUSTES.items() if v != ""},
               "rama": RAMA}, open(DESTINO / f"{nombre}.info.json", "w"), indent=1)
    print(f"  {nombre}: {minutos:.0f} min")
    subprocess.run(["ollama", "rm", etiqueta], env=env_ollama, capture_output=True)   # libera disco

# Tabla final
filas = ["| modelo | total /80 | cerradas | citas | abstencion | juez | min |", "|---|---|---|---|---|---|---|"]
for ev in sorted(DESTINO.glob("*.eval.json")):
    nombre = ev.name[:-len(".eval.json")]
    try:
        r = json.load(open(ev))
        info = json.load(open(DESTINO / f"{nombre}.info.json")) if (DESTINO / f"{nombre}.info.json").exists() else {}
        juez = r.get("correccion_ragas", {}).get("puntos", "-")
        filas.append(f"| {nombre} | {r['total_automatico']['obtenidos']} | {r['cerradas']['puntos']} | "
                     f"{r['citas']['puntos']} | {r['abstencion']['puntos']} | {juez} | {info.get('minutos', '-')} |")
    except Exception as e:
        filas.append(f"| {nombre} | (no se pudo leer: {e}) |")
tabla = "\n".join(filas)
(DESTINO / "resumen.md").write_text("# Comparacion de modelos en sample_50\n\n" + tabla + "\n", encoding="utf-8")
print(tabla)
print("\nReferencias: primer grafo con qwen3:8b-q8_0 = 49,75/80; sistema anterior 47,05/80")